In [0]:
import random
import uuid
import yaml

from config import DeployConfig
from itertools import product

In [0]:
dbutils.widgets.text("config_path", "./config/env_variables.yml")
dbutils.widgets.text("pet_features_path", "./config/pet_features.yml")
config_path = dbutils.widgets.get("config_path")
pet_features_path = dbutils.widgets.get("pet_features_path")
cfg = DeployConfig.from_yaml(config_path)

In [0]:
user_table = getattr(cfg, f"user_table")

# USER DATA

In [0]:
with open(pet_features_path, "r") as file:
    pet_features = yaml.safe_load(file)

In [0]:
num_rows = 1000

pet_types = list(pet_features["pets"].keys())

unique_combinations = set()
for pet_type in pet_types:
    pet_data = pet_features["pets"][pet_type]
    breeds = pet_data["breeds"]
    temperaments = pet_data["temperaments"]
    max_age = pet_data["max_age"]

    for combo in product([pet_type], range(1, max_age+1), ["Male", "Female"], breeds, temperaments):
      unique_combinations.add(combo)
    
if len(unique_combinations) < num_rows:
  print(f"Warning: Only {len(unique_combinations)} unique combinations available, but {num_rows} rows were requested.")
  print(f"Generating {len(unique_combinations)} rows instead.")

sampled_data = random.sample(list(unique_combinations), min(num_rows, len(unique_combinations)))

data = [(str(uuid.uuid4()), *row) for row in sampled_data]

columns = ["user_id", "pet_type", "age", "sex", "breed", "temperament"]
df_pet_features = spark.createDataFrame(data, columns)

In [0]:
display(df_pet_features)

In [0]:
df_pet_features.write.format("delta").mode("overwrite").saveAsTable(user_table.path)